# Step-Back Prompting [Step 4 - Zoom Out Before You Search]

> **MLCourse - Agentic AI - Advanced RAG - Query Transformation**

**Step-back prompting** (Zheng et al. 2023) handles a failure the previous two
techniques do not touch: the question is **too specific** for the index.

The user asks about a detail. The corpus explains the detail only inside a
broader passage that never uses the detail's words. Retrieval on the specific
question finds fragments; retrieval on the *general concept behind it* finds the
passage that actually explains things.

The move is to generate a **step-back question** - a deliberately broader
version - retrieve for both, and give the LLM the specific evidence *and* the
background it needs to reason with.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

# Paragraph-sized chunks: human-readable units, good enough for retrieval demos
# and identical to the chunking used in ../01_hybrid_search.
paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]

print("characters :", len(raw_text))
print("paragraphs :", len(paragraphs))
print("example    :", paragraphs[10][:150], "...")

characters : 144696
paragraphs : 237
example    : Alice was not a bit hurt, and she jumped up on to her feet in a moment: she looked up, but it was all dark overhead; before her was another long passa ...


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_vectors = encoder.encode(paragraphs, normalize_embeddings=True,
                             batch_size=64, show_progress_bar=False)


def embed(text):
    return encoder.encode([text], normalize_embeddings=True)[0]


def dense_rank(text, top_n=10):
    """Rank paragraph indices by cosine similarity to `text` (best first)."""
    sims = doc_vectors @ embed(text)
    return [int(i) for i in np.argsort(sims)[::-1][:top_n]]


def dense_scores(text):
    return doc_vectors @ embed(text)


print("dense index ready:", doc_vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

dense index ready: (237, 384)


### 2. What a step-back question looks like

The transformation is "abstract one level up". A few examples of the shape:

| original (specific) | step-back (general) |
|---|---|
| "Why did Alice's neck grow so long after the cake?" | "How does Alice's body change size in the story?" |
| "What did the Dormouse say about treacle?" | "What happens during the mad tea party?" |
| "Which card painted the roses red?" | "What is the Queen's garden like and who serves her?" |

Notice what is preserved and what is dropped: the **entity and setting** stay,
the **narrow detail** goes. A step-back question that drops the entity too
("what is a story?") is useless - that is the main failure mode to prompt
against.

In [5]:
STEP_BACK_PROMPT = (
    "You turn a specific question into a broader 'step-back' question that asks "
    "about the general concept or scene behind it.\n\n"
    "Rules:\n"
    "- Keep the main characters, objects and setting.\n"
    "- Drop the narrow detail being asked about.\n"
    "- Stay about the same topic - do not become generic.\n"
    "- Output ONLY the step-back question, nothing else.\n\n"
    "Specific question: {question}\n\nStep-back question:"
)


def step_back(question):
    return ask(STEP_BACK_PROMPT.format(question=question)).strip().strip('"')


SPECIFIC_QUESTIONS = [
    "Why did Alice's neck grow so long?",
    "What did the Dormouse say about the treacle well?",
    "Which gardeners were painting the white roses red?",
]

pairs = []
for q in SPECIFIC_QUESTIONS:
    sb = step_back(q)
    pairs.append((q, sb))
    print(f"specific : {q}")
    print(f"step-back: {sb}\n")

specific : Why did Alice's neck grow so long?
step-back: What happened to Alice's body in Wonderland?



specific : What did the Dormouse say about the treacle well?
step-back: What did the Dormouse say?



specific : Which gardeners were painting the white roses red?
step-back: Who were the gardeners painting the roses?



### 3. What each question retrieves

The point is not that the step-back question is *better*. It is that it
retrieves **different, complementary** material - broader scene-setting passages
rather than the exact sentence.

In [6]:
specific_q, general_q = pairs[0]

spec_ids = dense_rank(specific_q, top_n=4)
gen_ids = dense_rank(general_q, top_n=4)

print("SPECIFIC:", specific_q)
for rank, doc_id in enumerate(spec_ids, 1):
    print(f"  #{rank} doc_{doc_id}: {paragraphs[doc_id][:105]}...")

print("\nSTEP-BACK:", general_q)
for rank, doc_id in enumerate(gen_ids, 1):
    tag = "also in specific" if doc_id in spec_ids else "NEW"
    print(f"  #{rank} doc_{doc_id} ({tag}): {paragraphs[doc_id][:105]}...")

print(f"\noverlap: {len(set(spec_ids) & set(gen_ids))} of 4 - the two questions "
      f"are pulling from different parts of the corpus")

SPECIFIC: Why did Alice's neck grow so long?
  #1 doc_96: “Come, my head’s free at last!” said Alice in a tone of delight, which changed into alarm in another mome...
  #2 doc_209: Just at this moment Alice felt a very curious sensation, which puzzled her a good deal until she made out...
  #3 doc_93: This time Alice waited patiently until it chose to speak again. In a minute or two the Caterpillar took t...
  #4 doc_167: Alice did not much like keeping so close to her: first, because the Duchess was _very_ ugly; and secondly...

STEP-BACK: What happened to Alice's body in Wonderland?
  #1 doc_209 (also in specific): Just at this moment Alice felt a very curious sensation, which puzzled her a good deal until she made out...
  #2 doc_153 (NEW): Alice began to feel very uneasy: to be sure, she had not as yet had any dispute with the Queen, but she k...
  #3 doc_96 (also in specific): “Come, my head’s free at last!” said Alice in a tone of delight, which changed into alarm in another mome

### 4. The combined retrieval

Step-back retrieval is a **union**, not a replacement. You want both: the
specific evidence *and* the surrounding context. A simple interleave keeps the
specific results first, which matters because LLMs weight early context more
heavily.

In [7]:
def step_back_retrieve(question, k_specific=3, k_general=2):
    """Retrieve for the question AND for its step-back version."""
    general = step_back(question)
    spec = dense_rank(question, top_n=k_specific)
    gen = [d for d in dense_rank(general, top_n=k_general + k_specific)
           if d not in spec][:k_general]
    return general, spec, gen


general_q, spec_ids, gen_ids = step_back_retrieve(
    "What did the Dormouse say about the treacle well?")

print("step-back question:", general_q)
print("\nspecific evidence:")
for doc_id in spec_ids:
    print(f"  doc_{doc_id}: {paragraphs[doc_id][:100]}...")
print("\nbackground context:")
for doc_id in gen_ids:
    print(f"  doc_{doc_id}: {paragraphs[doc_id][:100]}...")

step-back question: What did the Dormouse say?

specific evidence:
  doc_136: The Dormouse had closed its eyes by this time, and was going off into a doze; but, on being pinched ...
  doc_126: There was a table set out under a tree in front of the house, and the March Hare and the Hatter were...
  doc_133: Alice did not quite know what to say to this: so she helped herself to some tea and bread-and-butter...

background context:
  doc_137: This piece of rudeness was more than Alice could bear: she got up in great disgust, and walked off; ...
  doc_57: This speech caused a remarkable sensation among the party. Some of the birds hurried off at once: on...


### 5. Does the extra context change the answer?

Here is the honest test: same question, answered from specific-only context and
from combined context.

In [8]:
QUESTION = "What did the Dormouse say about the treacle well?"


def generate(doc_ids, question):
    context = "\n\n".join(f"[doc_{i}] {paragraphs[i]}" for i in doc_ids)
    return ask(
        "Answer the question using ONLY the context below. Be specific and "
        "quote what you relied on. If the context lacks the answer, say so.\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"
    )


print("=" * 70)
print("A) specific retrieval only, docs", spec_ids)
print("=" * 70)
print(generate(spec_ids, QUESTION))

A) specific retrieval only, docs [136, 126, 133]


The provided context does not contain the answer to this question.

The context includes:
*   [doc_136]: The Dormouse speaking about words beginning with "M" (mouse-traps, moon, memory, muchness).
*   [doc_126]: A description of the Dormouse sleeping between the March Hare and the Hatter.
*   [doc_133]: Alice asking the Dormouse, “Why did they live at the bottom of a well?”

However, none of the documents contain the Dormouse's response regarding the "treacle well."


In [9]:
print("=" * 70)
print("B) specific + step-back context, docs", spec_ids + gen_ids)
print("=" * 70)
print(generate(spec_ids + gen_ids, QUESTION))

B) specific + step-back context, docs [136, 126, 133, 137, 57]


The provided context does not contain the answer to this question.

While the context mentions the Dormouse discussing things that begin with an "M" (such as "mouse-traps, and the moon, and memory, and muchness") in [doc_136], and Alice asking, "Why did they live at the bottom of a well?" in [doc_133], the specific response from the Dormouse regarding the "treacle well" is not included in the provided documents.


### 6. Step-back reasoning, not just step-back retrieval

The original paper does something slightly more ambitious than retrieval: it
asks the model to **answer the general question first**, then use that as
grounding to answer the specific one. This is a two-hop reasoning chain, and it
helps most when the specific answer must be *derived* rather than looked up.

In [10]:
def step_back_reasoning(question):
    general = step_back(question)
    gen_ctx = "\n\n".join(f"[doc_{i}] {paragraphs[i]}"
                           for i in dense_rank(general, top_n=3))
    principle = ask(
        "Using ONLY the context, answer the general question in 2-3 sentences.\n\n"
        f"Context:\n{gen_ctx}\n\nGeneral question: {general}\nAnswer:"
    )
    spec_ctx = "\n\n".join(f"[doc_{i}] {paragraphs[i]}"
                            for i in dense_rank(question, top_n=3))
    final = ask(
        "You have established background knowledge and specific passages. "
        "Answer the specific question using both. Stay grounded in the passages.\n\n"
        f"Background you established:\n{principle}\n\n"
        f"Specific passages:\n{spec_ctx}\n\n"
        f"Specific question: {question}\nAnswer:"
    )
    return general, principle, final


general, principle, final = step_back_reasoning("Why did Alice's neck grow so long?")

print("step-back question :", general)
print("\nestablished background:")
print(" ", principle)
print("\nfinal grounded answer:")
print(" ", final)

step-back question : What happened to Alice's body in Wonderland?

established background:
  Alice experienced several bizarre physical transformations in Wonderland, including growing larger, having her head become free while her shoulders disappeared, and developing an immense length of neck. These changes caused her considerable confusion and alarm as she struggled to understand her shifting form.

final grounded answer:
  Based on the provided passages, Alice's neck grew so long because she was undergoing a physical transformation where she was "beginning to grow larger again" [doc_209]. This growth was a result of consuming parts of a mushroom, as the Caterpillar had previously explained that "One side will make you grow taller, and the other side will make you grow shorter" [doc_93]. The specific manifestation of this growth in this instance was her head becoming free while her shoulders disappeared, leaving her with an "immense length of neck" [doc_96].


### 7. When step-back helps, and when it does not

**Helps** when:

- The question asks about a detail embedded in a larger narrative or process.
- The corpus organises information by topic, not by fact.
- The answer requires reasoning over background, not a lookup.

**Hurts** when:

- The question is already broad - stepping back gives you a vague query that
  retrieves everything and discriminates nothing.
- The question is a precise lookup (an identifier, a date, a code). Broadening
  it actively destroys the signal.
- Your corpus is a flat list of independent facts with no hierarchy to step
  back into.

The routing lesson: step-back is a **conditional** transformation. A production
system should decide per query whether to apply it - which is the kind of choice
[`../06_adaptive_rag`](../06_adaptive_rag/README.md) teaches how to make.

### 8. Relationship to decomposition

Step-back goes **up** the abstraction ladder: one question becomes one broader
question. Decomposition goes **sideways**: one compound question becomes several
independent sub-questions, each answered separately and then combined.

They fix different problems, and this course teaches decomposition in its
natural agentic home:
[`../03_agentic_rag/03_query_decomposition.ipynb`](../03_agentic_rag/03_query_decomposition.ipynb).
Read it after this notebook - together they cover the full transformation space.

### 9. Key takeaways

- Step-back prompting generates a **broader** question and retrieves for both.
- Use the results as a **union**: specific evidence first, background second.
- The stronger variant answers the general question first, then uses that as
  grounding - a two-hop chain.
- It is conditional: broad questions and precise lookups are made worse by it.

Next: [`05_comparing_transformations.ipynb`](05_comparing_transformations.ipynb)
puts all of these on the same evaluation set.